###Loading data.

In [ ]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
path = kagglehub.competition_download('filament-segmentation-2026')

print("Path to competition files:", path)

100%|██████████| 671M/671M [00:04<00:00, 148MB/s]

Extracting files...


Path to competition files: /root/.cache/kagglehub/competitions/filament-segmentation-2026


In [ ]:
def generate_mask_for_image(img_info, filaments):
    """
    It creates a binary mask (0 and 1).
    """
    height = img_info['height']
    width = img_info['width']

    # Background
    mask = np.zeros((height, width), dtype=np.uint8)

    for ann in filaments:
        segmentations = ann.get('segmentation', [])

        for seg in segmentations:
            poly = np.array(seg, dtype=np.int32).reshape((-1, 2))

            cv2.fillPoly(mask, [poly], color=1)

    return mask


In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import Sequence

class FilamentDataGenerator(Sequence):
    def __init__(self, images_list, annotations_map, img_dir, batch_size=8, target_size=(512, 512), shuffle=True):
        self.images_list = images_list
        self.annotations_map = annotations_map
        self.img_dir = img_dir
        self.batch_size = batch_size
        self.target_size = target_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.images_list))
        self.on_epoch_end()

    def __len__(self):
      """
      Number of batch per epoch.
      """
      return int(np.ceil(len(self.images_list) / self.batch_size))

    def on_epoch_end(self):
      """
      Randomly shuffle the data for each epoch.
      """
      if self.shuffle:
          np.random.shuffle(self.indexes)

    def __getitem__(self, index):
      """
      Reads and resizes 8 photos and smoothes them.
      """

      batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]

      batch_images = []
      batch_masks = []

      for idx in batch_indexes:
            img_info = self.images_list[idx]
            img_id = img_info['id']

            # Reading an image from disk.
            img_path = os.path.join(self.img_dir, img_info['file_name'])
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # Generating a binary mask of the same image.
            filaments = self.annotations_map.get(img_id, [])
            mask = generate_mask_for_image(img_info, filaments)

            # On-the-fly resizing.
            image_resized = cv2.resize(image, self.target_size, interpolation=cv2.INTER_LINEAR)
            mask_resized = cv2.resize(mask, self.target_size, interpolation=cv2.INTER_NEAREST)

            # Data normalization (numbers between 0 and 1).
            image_normalized = image_resized.astype(np.float32) / 255.0
            mask_normalized = np.expand_dims(mask_resized.astype(np.float32), axis=-1)

            batch_images.append(image_normalized)
            batch_masks.append(mask_normalized)

      return np.array(batch_images), np.array(batch_masks)

In [ ]:
from sklearn.model_selection import train_test_split
import os
import json
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Path
BASE_DIR = '/root/.cache/kagglehub/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026'
ANN_PATH = os.path.join(BASE_DIR, 'train', 'MAGFiLO_1.0_Annotations_kaggle2026_train.json')
# Reading json path
with open(ANN_PATH, 'r') as f:
    coco_data = json.load(f)
all_images = coco_data['images']

train_images, val_images = train_test_split(all_images, test_size=0.25, random_state=19)
IMG_DIR = os.path.join(BASE_DIR, 'train', 'train_images')

from collections import defaultdict

filaments_by_image = defaultdict(list)
for ann in coco_data['annotations']:
    filaments_by_image[ann['image_id']].append(ann)

train_generator = FilamentDataGenerator(
    images_list=train_images,
    annotations_map=filaments_by_image,
    img_dir=IMG_DIR,
    batch_size=8,
    target_size=(512, 512),
    shuffle=True
)

val_generator = FilamentDataGenerator(
    images_list=val_images,
    annotations_map=filaments_by_image,
    img_dir=IMG_DIR,
    batch_size=8,
    target_size=(512, 512),
    shuffle=False
)

print(f"Number of batch Train: {len(train_generator)}")
print(f"Number of batch Validation: {len(val_generator)}")

Number of batch Train: 109
Number of batch Validation: 37


In [ ]:
from keras.layers import Dense, Conv2D, MaxPooling2D, BatchNormalization, Dropout, Flatten, Conv2DTranspose, concatenate, Input
from keras.models import Sequential
from keras.models import Model

def build_unet(input_shape=(512, 512, 3)):
    inputs = Input(input_shape)

    # Encoder block 1
    c1 = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    b1 = BatchNormalization()(c1)
    p1 = MaxPooling2D((2, 2))(b1)

    # Encoder block 2
    c2 = Conv2D(64, (3, 3), activation='relu', padding='same')(p1)
    b2 = BatchNormalization()(c2)
    p2 = MaxPooling2D((2, 2))(b2)

    # Encoder block 3
    c3 = Conv2D(128, (3, 3), activation='relu', padding='same')(p2)
    b3 = BatchNormalization()(c3)
    p3 = MaxPooling2D((2, 2))(b3)

    # Central Block
    c4 = Conv2D(256, (3, 3), activation='relu', padding='same')(p3)
    c5 = Conv2D(128, (3, 3), activation='relu', padding='same')(c4)
    c6 = Conv2D(128, (3, 3), activation='relu', padding='same')(c5)

    # Decoder block 1
    t1 = Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c6)
    t1 = concatenate([t1, b3])
    c7 = Conv2D(64, (3, 3), activation='relu', padding='same')(t1)
    c8 = Conv2D(64, (3, 3), activation='relu', padding='same')(c7)

    # Decoder block 2
    t2 = Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c8)
    t2 = concatenate([t2, b2])
    c9 = Conv2D(64, (3, 3), activation='relu', padding='same')(t2)
    c10 = Conv2D(32, (3, 3), activation='relu', padding='same')(c9)

    # Decoder block 3
    t3 = Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c10)
    t3 = concatenate([t3, b1])
    c11 = Conv2D(64, (3, 3), activation='relu', padding='same')(t3)
    c12 = Conv2D(32, (3, 3), activation='relu', padding='same')(c11)

    outputs = Conv2D(1, (1, 1), activation='sigmoid')(c12)
    model = Model(inputs=[inputs], outputs=[outputs])

    return model

In [ ]:
model = build_unet()
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 512, 512,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 512, 512,  │        896 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 512, 512,  │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 256, 256,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 128, 128,  │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 64, 64,    │    295,168 │ max_pooling2d_2[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 64, 64,    │    295,040 │ conv2d_3[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 64, 64,    │    147,584 │ conv2d_4[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose    │ (None, 128, 128,  │     65,664 │ conv2d_5[0][0]    │
│ (Conv2DTranspose)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 128, 128,  │          0 │ conv2d_transpose… │
│ (Concatenate)       │ 256)              │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 128, 128,  │    147,520 │ concatenate[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 128, 128,  │     36,928 │ conv2d_6[0][0]  

 Total params: 1,371,297 (5.23 MB)

 Trainable params: 1,370,849 (5.23 MB)

 Non-trainable params: 448 (1.75 KB)

In [ ]:
import tensorflow.keras.backend as K

def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = 1.0 - dice_coef(y_true, y_pred)
    return bce + dice

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=bce_dice_loss,
    metrics=[dice_coef, 'binary_accuracy']
)


In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
    )

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 185s 1s/step - binary_accuracy: 0.9686 - dice_coef: 0.0408 - loss: 1.0595 - val_binary_accuracy: 0.9963 - val_dice_coef: 0.0062 - val_loss: 1.1254
Epoch 2/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 76s 696ms/step - binary_accuracy: 0.9964 - dice_coef: 0.4741 - loss: 0.5476 - val_binary_accuracy: 0.9963 - val_dice_coef: 0.0048 - val_loss: 1.0186
Epoch 3/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 76s 694ms/step - binary_accuracy: 0.9969 - dice_coef: 0.5615 - loss: 0.4585 - val_binary_accuracy: 0.9963 - val_dice_coef: 0.0081 - val_loss: 1.0225
Epoch 4/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 76s 696ms/step - binary_accuracy: 0.9971 - dice_coef: 0.5918 - loss: 0.4270 - val_binary_accuracy: 0.9960 - val_dice_coef: 0.3400 - val_loss: 0.6794
Epoch 5/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 76s 693ms/step - binary_accuracy: 0.9972 - dice_coef: 0.6101 - loss: 0.4083 - val_binary_accuracy: 0.9970 - val_dice_coef: 0.3748 - val_loss: 0.6503
Epoch 6/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 76s 699ms/step - 

In [ ]:
history = history.history
print(history.keys())

dict_keys(['binary_accuracy', 'dice_coef', 'loss', 'val_binary_accuracy', 'val_dice_coef', 'val_loss'])


In [ ]:
import plotly.express as px
import plotly
import plotly.io as pio

epochs = list(range(1, len(history['loss']) + 1))

fig_loss = px.line(
    x=epochs,
    y=[history['loss'], history['val_loss']],
    labels={'x': 'Epoch', 'value': 'Loss', 'variable': 'Dataset'},
    title='Loss Curve',
    markers=True
)
fig_loss.show()

In [ ]:
fig_dice = px.line(
    x=epochs,
    y=[history['dice_coef'], history['val_dice_coef']],
    labels={'x': 'Epoch', 'value': 'Dice Score', 'variable': 'Dataset'},
    title='Dice Score',
    markers=True
)
fig_dice.show()

In [ ]:
for i in range(9)
  val_generator = FilamentDataGenerator(
    images_list=val_images,
    annotations_map=filaments_by_image,
    img_dir=IMG_DIR,
    batch_size=1,
    target_size=(512, 512),
    shuffle=False
  )

prediction = model.predict(choice)

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning:

Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.



289/289 ━━━━━━━━━━━━━━━━━━━━ 13s 39ms/step
